In [ ]:
from datasets import load_dataset

dataset = load_dataset("Linq-AI-Research/FinDER")

# The authors provide the 'corpus' as a unique set of all 'references' 
# combined with distracting passages.
full_corpus = set()
for row in dataset['train']:
    for passage in row['references']:
        full_corpus.add(passage)

print(f"Unique passage count: {len(full_corpus)}")

In [ ]:
"""
Quick snippet for Jupyter notebook - Token estimation for 10-K documents
Copy this into a notebook cell to run
"""

# === PASTE THIS INTO JUPYTER NOTEBOOK ===

import os
from html_parser import parse_ixbrl_html 

# Try tiktoken, fallback to rough estimate
try:
    import tiktoken 
    enc = tiktoken.get_encoding("cl100k_base")
    count_tokens = lambda text: len(enc.encode(text))
    print("✓ Using tiktoken (accurate)")
except:
    count_tokens = lambda text: len(text) // 4
    print("⚠ Using rough estimate (chars/4)")

# Quick sample of 10 documents
sample_files = [f"10k/{f}" for f in os.listdir("10k") if f.endswith('.html')][:50]

total_tokens = 0
for file_path in sample_files:
    parsed = parse_ixbrl_html(file_path)
    tokens = count_tokens(parsed['text'])
    total_tokens += tokens
    print(f"{parsed['ticker']:8} {tokens:>8,} tokens  ({len(parsed['text']):>9,} chars)")

# Extrapolate to all 497 files
avg_tokens = total_tokens / len(sample_files)
all_files_count = len([f for f in os.listdir("10k") if f.endswith('.html')])
estimated_total = avg_tokens * all_files_count

print(f"\n{'='*50}")
print(f"Sample average:      {avg_tokens:>15,.0f} tokens/doc")
print(f"Total files:         {all_files_count:>15,}")
print(f"Estimated total:     {estimated_total:>15,.0f} tokens")
print(f"Estimated cost:      ${(estimated_total/1000)*0.0001:>15.2f} (OpenAI ada)")
print(f"{'='*50}")




In [ ]:
"""
Parse all 10-K HTML files and save as JSON
Test on 1 file first, then process all
"""
import os
import json
from html_parser import parse_ixbrl_html
from tqdm import tqdm

HTML_DIR = "10k"
OUTPUT_FILE = "parsed_10k_documents.json"

def test_single_file():
    """Test on one file first"""
    html_files = [f for f in os.listdir(HTML_DIR) if f.endswith('.html')]

    if not html_files:
        print("No HTML files found!")
        return None

    # Test on first file
    test_file = html_files[0]
    test_path = os.path.join(HTML_DIR, test_file)

    print(f"Testing parser on: {test_file}")
    print("="*60)

    try:
        parsed = parse_ixbrl_html(test_path)

        print(f"Ticker:          {parsed['ticker']}")
        print(f"Company Name:    {parsed['metadata'].get('company_name', 'N/A')}")
        print(f"Period End:      {parsed['metadata'].get('period_end_date', 'N/A')}")
        print(f"Text Length:     {len(parsed['text']):,} characters")
        print(f"\nFirst 300 chars of text:")
        print("-"*60)
        print(parsed['text'][:300] + "...")
        print("-"*60)
        print("\n✓ Test successful!")

        return parsed

    except Exception as e:
        print(f"✗ Error parsing file: {e}")
        return None

def parse_all_files(limit=None):
    """Parse all HTML files in directory"""
    html_files = [f for f in os.listdir(HTML_DIR) if f.endswith('.html')]

    if limit:
        html_files = html_files[:limit]

    print(f"\nParsing {len(html_files)} HTML files...")

    parsed_docs = {}
    errors = []

    for html_file in tqdm(html_files, desc="Parsing"):
        file_path = os.path.join(HTML_DIR, html_file)

        try:
            parsed = parse_ixbrl_html(file_path)
            ticker = parsed['ticker']

            # Store by ticker
            parsed_docs[ticker] = {
                'ticker': ticker,
                'company_name': parsed['metadata'].get('company_name', ''),
                'period_end_date': parsed['metadata'].get('period_end_date', ''),
                'source_file': html_file,
                'text': parsed['text'],
                'text_length': len(parsed['text'])
            }

        except Exception as e:
            errors.append({'file': html_file, 'error': str(e)})

    print(f"\n✓ Successfully parsed: {len(parsed_docs)} documents")
    if errors:
        print(f"✗ Errors: {len(errors)} files failed")
        for err in errors[:5]:  # Show first 5 errors
            print(f"  - {err['file']}: {err['error']}")

    return parsed_docs, errors

def save_parsed_docs(parsed_docs, output_file):
    """Save parsed documents as JSON"""
    print(f"\nSaving to {output_file}...")

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(parsed_docs, f, indent=2, ensure_ascii=False)

    print(f"✓ Saved {len(parsed_docs)} documents")

    # Print summary stats
    total_chars = sum(doc['text_length'] for doc in parsed_docs.values())
    avg_chars = total_chars / len(parsed_docs)

    print(f"\nSummary:")
    print(f"  Total documents:  {len(parsed_docs)}")
    print(f"  Total characters: {total_chars:,}")
    print(f"  Avg per doc:      {avg_chars:,.0f} chars")

if __name__ == "__main__":
    # Step 1: Test on single file
    print("STEP 1: Testing on single file")
    print("="*60)
    test_result = test_single_file()

    if test_result:
        print("\n" + "="*60)
        user_input = input("\nTest successful! Parse all files? (y/n): ").lower()

        if user_input == 'y':
            # Step 2: Parse all files
            print("\nSTEP 2: Parsing all files")
            print("="*60)
            parsed_docs, errors = parse_all_files()

            # Step 3: Save results
            save_parsed_docs(parsed_docs, OUTPUT_FILE)

            # Save errors if any
            if errors:
                with open("parsing_errors.json", 'w') as f:
                    json.dump(errors, f, indent=2)
                print(f"✓ Error log saved to parsing_errors.json")
        else:
            print("Skipped full parsing.")
    else:
        print("\n✗ Test failed. Fix errors before parsing all files.")
